# Cricket Research Answer Bot — RAG Capstone

A Retrieval-Augmented Generation (RAG) chatbot that answers questions over a curated set of
cricket research papers. This notebook walks through the pipeline step by step, explaining
the reasoning behind each design choice — treat it as the main deliverable alongside `README.md`.

```
PDFs  ->  load & chunk  ->  embed (2 models)  ->  ChromaDB  ->  retrieve (2 strategies)  ->  LLM  ->  grounded answer + citations
```


## Setup

Run once. Requires `GEMINI_API_KEY` in a `.env` file at the project root for Steps 3 onward.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv("../.env")

import warnings
warnings.filterwarnings("ignore")

print("Setup complete.")


## Step 1 — Data Loading

We load every PDF in `data/papers/` using `PyPDFLoader`, which returns one `Document` per
**page** with the page number already attached to metadata — exactly the granularity we need
to cite "paper title, page N" later.

If `data/papers/` is empty, first run `python data/generate_sample_papers.py` to create the
3 sample cricket papers (no internet required).

In [ ]:
from src.data_loading import load_all_papers, inspect_documents

docs = load_all_papers()
inspect_documents(docs)


## Step 2 — Chunking

`RecursiveCharacterTextSplitter` with `chunk_size=700`, `chunk_overlap=120`. It tries
paragraph breaks first, then lines, then sentences — only falling back to a hard character
cut as a last resort, so chunks stay coherent instead of being sliced mid-sentence.

- **Chunk too small** -> loses surrounding context; the LLM may misread an isolated sentence.
- **Chunk too large** -> mixes multiple ideas into one embedding, diluting it and hurting
  retrieval precision.
- **Overlap** prevents losing a sentence that falls exactly on a chunk boundary.

In [ ]:
from src.chunking import chunk_documents, inspect_chunks

chunks = chunk_documents(docs)
inspect_chunks(chunks)


## Step 3 — Embeddings

Two models, compared as required by the brief:

| Model | Type | Notes |
|---|---|---|
| `BAAI/bge-m3` | Open-source, local | Free, no API key, needs local compute |
| `models/text-embedding-004` | Commercial, Gemini API | Fast, small per-token cost |

Both are wrapped behind the same interface, so the rest of the pipeline doesn't care which
is active. We build **one Chroma collection per embedding model** so the two can be compared
side by side without overwriting each other.

In [ ]:
from src.embeddings import get_hf_embeddings, get_gemini_embeddings

hf_embeddings = get_hf_embeddings()
print("Loaded open-source embedding model: BAAI/bge-m3")

# Requires GEMINI_API_KEY in .env
gemini_embeddings = get_gemini_embeddings()
print("Loaded commercial embedding model: models/text-embedding-004")


## Step 4 — Vector Database (ChromaDB)

We index the same chunks into two separate Chroma collections, one per embedding model.

In [ ]:
from src.vectorstore import build_vectorstore, validate_retrieval
from src.config import HF_COLLECTION_NAME, GEMINI_COLLECTION_NAME

vectorstore_hf = build_vectorstore(chunks, hf_embeddings, collection_name=HF_COLLECTION_NAME)
vectorstore_gemini = build_vectorstore(chunks, gemini_embeddings, collection_name=GEMINI_COLLECTION_NAME)

print("Both collections indexed.\n")
validate_retrieval(vectorstore_hf, "How does hybrid retrieval combine BM25 and dense search?")


### Comparing the two embedding models

A quick qualitative check: for a handful of representative queries, does each embedding
model surface the correct passage as (or near) its #1 result?

In [ ]:
test_queries = [
    "What is the acute:chronic workload ratio?",
    "Why do teams send power-hitters to open the batting in T20s?",
    "What makes a mixed bowling action risky?",
]

for q in test_queries:
    print("=" * 70)
    print(f"bge-m3 (open-source) -> {q!r}")
    validate_retrieval(vectorstore_hf, q, k=2)
    print()
    print(f"text-embedding-004 (commercial) -> {q!r}")
    validate_retrieval(vectorstore_gemini, q, k=2)
    print()


## Step 5 — Retrieval Strategies

1. **Dense retrieval (cosine similarity)** — the baseline.
2. **Hybrid retrieval (BM25 + dense)** — combines keyword scoring (BM25) with dense
   similarity via rank fusion. This catches specific technical terms (e.g. "powerplay",
   "acute:chronic workload ratio") that pure semantic search can sometimes rank lower.

We use the **Gemini-embedded** collection going forward as the primary vectorstore, since
it is the commercial embedding option specified for this build.

In [ ]:
from src.retrieval import get_dense_retriever, get_hybrid_retriever, compare_retrievers

dense_retriever = get_dense_retriever(vectorstore_gemini, k=3)
hybrid_retriever = get_hybrid_retriever(vectorstore_gemini, chunks, k=3)

compare_retrievers("What is the acute:chronic workload ratio and why is it tracked?", dense_retriever, hybrid_retriever)


In [ ]:
compare_retrievers("Why is bowling variation important in the death overs?", dense_retriever, hybrid_retriever)


**Observation:** hybrid retrieval consistently pulls in chunks containing the query's exact
technical term (e.g. "acute:chronic workload ratio", "death overs") even when dense-only
retrieval ranks a more loosely related passage higher. We select **hybrid retrieval** as the
final strategy for the app.

## Step 6 — RAG Chain ("the brain")

Wired together with LangChain's LCEL (`|` pipe syntax):

```
retriever -> format chunks into labeled context -> prompt -> Gemini -> answer
```

Key prompt engineering decisions:
- Answer **only** from the retrieved context, never from the model's own trained knowledge.
- Say **"I don't know"** when context is insufficient, rather than guess.
- Label every chunk with its paper title + page number inside the prompt itself.
- `temperature=0.0` for deterministic, grounded answers.

In [ ]:
from src.rag_chain import ask

result = ask(
    "What is the acute:chronic workload ratio and why is it tracked?",
    hybrid_retriever,
    model_name="3.1-flash",
)

print("ANSWER:\n", result["answer"])
print("\nSOURCES:")
for s in result["sources"]:
    print(f"  - {s['paper_title']} (p.{s['page_number']}): {s['snippet']}...")


## Step 7 — Testing & Evaluation

10 test questions spanning all 3 papers, including one deliberately out-of-scope question
("What is the capital of France?") to verify the system correctly refuses rather than
hallucinating an answer.

In [ ]:
test_questions = [
    "What context-aware metrics are used instead of raw batting average?",
    "What inputs does a win probability model typically use?",
    "Why do win probabilities swing faster in T20 than in ODIs or Tests?",
    "Which model types are commonly used for player performance prediction and why?",
    "What is the delivery stride phase in fast bowling and why does it matter?",
    "Why is a mixed bowling action associated with higher injury risk?",
    "What is the acute:chronic workload ratio and why is it tracked?",
    "Why is bowling variation important in the death overs?",
    "How do teams use matchup analytics during a T20 innings?",
    "What is the capital of France?",  # deliberately out-of-scope
]

for q in test_questions:
    result = ask(q, hybrid_retriever, model_name="3.1-flash")
    print("=" * 70)
    print("Q:", q)
    print("A:", result["answer"])
    print()


## Step 8 — Stretch Goal: Streamlit UI

A Streamlit app (`app.py`) reuses every `src/` module used above — no logic duplicated
between notebook and app. It includes a sidebar toggle to switch between dense and hybrid
retrieval live, and a Gemini model-tier selector (`3.1-flash-lite` / `3.1-flash` / `3.1-pro`).

Run it with:
```bash
streamlit run app.py
```

## Conclusion

| Component | Choice |
|---|---|
| Loader | PyPDFLoader — per-page metadata for citation |
| Chunking | RecursiveCharacterTextSplitter, 700 / 120 |
| Embeddings | bge-m3 (open-source) + text-embedding-004 (commercial) |
| Vector DB | ChromaDB, local, metadata-native |
| Retrieval | Dense (cosine) + Hybrid (BM25 + dense) — hybrid selected |
| LLM | Google Gemini — toggle between 3.1-flash-lite / 3.1-flash / 3.1-pro |
| Stretch goal | Streamlit UI with live retrieval-strategy toggle |

**Future improvements:** cross-encoder re-ranking on top of hybrid retrieval, conversational
memory for multi-turn Q&A, RAGAS/DeepEval for automated repeatable scoring, and expanding the
corpus with real published cricket analytics / sports-science papers.